# Product Alternative Finder Model

## Goal
Find cheaper products similar to a given product using **TF-IDF + Cosine Similarity**

## How It Works
1. **Extract** product names from raw data
2. **Clean** text (remove special characters, convert to lowercase)
3. **Vectorize** using TF-IDF (convert text to numerical vectors)
4. **Search** for cheaper products with high similarity scores
5. **Rank** by similarity and show savings percentage

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer

## Step 1: Import Required Libraries

In [2]:
df = pd.read_csv('cleaned_data.csv')

## Step 2: Load & Explore Data
Load product data from CSV and check for duplicates

In [4]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.head()

,Product Name,Price,Type,Source,Product_Name_Clean,Price_Numeric
0,"Samsung Galaxy S26 5G (Black, 12GB RAM, 256GB ...","87,999",Mobile,Amazon,samsung galaxy s26 5g black 12gb ram 256gb sto...,87999.0
1,"Samsung Galaxy S26 Ultra 5G (Black, 12GB RAM, ...","1,59,999",Mobile,Amazon,samsung galaxy s26 ultra 5g black 12gb ram g51...,159999.0
2,"Redmi A4 5G (Sparkle Purple, 4GB RAM, 128GB St...","11,999",Mobile,Amazon,redmi a4 5g sparkle purple 4gb ram 128gb stora...,11999.0
3,"iQOO Z11x 5G (Prismatic Green, 6GB RAM, 128 GB...","18,998",Mobile,Amazon,iqoo z11x 5g prismatic green 6gb ram 128 gb st...,18998.0
4,"Samsung Galaxy M06 5G Mobile (Blazing Black, 6...","11,999",Mobile,Amazon,samsung galaxy m06 5g mobile blazing black 6gb...,11999.0


In [7]:
# Extract product name by:
# Step 1: Remove content inside parentheses
# Step 2: Split on comma or pipe, keep first part
# Step 3: Strip extra spaces

df['Product_Name_Extracted'] = df['Product Name'].apply(
    lambda x: re.split(r'[,|]', re.sub(r"\(.*?\)", "", x))[0].strip()
)

## Step 3: Extract Product Names
Remove noise from product names (parentheses, extra text) to get clean names

In [15]:
print("Full extracted name for row 1:")
print(repr(df['Product_Name_Extracted'].iloc[1]))
print("\nFirst 10 extracted product names:")
print(df['Product_Name_Extracted'].head(10).tolist())

Full extracted name for row 1:
'Samsung Galaxy S26 Ultra 5G  with Built-in Privacy Display'

First 10 extracted product names:
['Samsung Galaxy S26 5G', 'Samsung Galaxy S26 Ultra 5G  with Built-in Privacy Display', 'Redmi A4 5G', 'iQOO Z11x 5G', 'Samsung Galaxy M06 5G Mobile', 'Samsung Galaxy M07 Mobile', 'realme NARZO 90x 5G', 'Samsung Galaxy M17e 5G Mobile', 'Samsung Galaxy M17e 5G Mobile', 'iQOO Z10x 5G']


In [8]:
# Clean extracted product names for TF-IDF
def clean_text_for_tfidf(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

df['Product_Name_Clean'] = df['Product_Name_Extracted'].apply(clean_text_for_tfidf)
print("Cleaned product names (first 10):")
print(df[['Product_Name_Extracted', 'Product_Name_Clean']].head(10))

Cleaned product names (first 10):
                              Product_Name_Extracted  \
0                              Samsung Galaxy S26 5G   
1  Samsung Galaxy S26 Ultra 5G  with Built-in Pri...   
2                                        Redmi A4 5G   
3                                       iQOO Z11x 5G   
4                       Samsung Galaxy M06 5G Mobile   
5                          Samsung Galaxy M07 Mobile   
6                                realme NARZO 90x 5G   
7                      Samsung Galaxy M17e 5G Mobile   
8                      Samsung Galaxy M17e 5G Mobile   
9                                       iQOO Z10x 5G   

                                  Product_Name_Clean  
0                                 samsung galaxy s g  
1  samsung galaxy s ultra g with builtin privacy ...  
2                                          redmi a g  
3                                          iqoo zx g  
4                          samsung galaxy m g mobile  
5                  

## Step 4: Clean Product Names
Convert to lowercase and remove special characters (only keep letters and spaces)

In [8]:
# Apply TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Product_Name_Clean'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"\nTop features (terms):")
print(tfidf_vectorizer.get_feature_names_out())

TF-IDF Matrix shape: (3992, 50)

Top features (terms):
['5g' 'adidas' 'ai' 'air' 'battery' 'black' 'blue' 'camera' 'dji' 'fhd'
 'flipkart' 'galaxy' 'gen' 'godrej' 'gold' 'green' 'hd' 'hp' 'ifb'
 'jacket' 'jeans' 'kent' 'kg' 'kurta' 'laptop' 'lenovo' 'levis' 'lg'
 'midea' 'nike' 'oneplus' 'philips' 'place' 'polo' 'pro' 'purple' 'ram'
 'realme' 'red' 'samsung' 'shirt' 'silver' 'spf' 'ssd' 'storage' 'tab'
 'tee' 'tv' 'watch' 'white']


## Step 5: Convert Text to Numbers (TF-IDF Vectorization)
Transform product names into numerical vectors for similarity comparison
- TF-IDF = Term Frequency-Inverse Document Frequency
- Creates a 176×50 matrix (176 products, 50 features)

In [9]:

# Model to find cheaper similar products
from sklearn.metrics.pairwise import cosine_similarity

def find_cheaper_alternatives(product_name, top_n=5):
    """
    Find cheaper products similar to the input product.
    
    Parameters:
    -----------
    product_name : str
        The product name to search for (e.g., 'iQOO Z11x 5G')
    top_n : int
        Number of cheaper alternatives to return
    
    Returns:
    --------
    DataFrame with cheaper similar products
    """
    
    # Find the product in dataset (case-insensitive)
    matching_products = df[df['Product_Name_Extracted'].str.contains(product_name, case=False, na=False)]
    
    if matching_products.empty:
        return f"Product '{product_name}' not found in dataset"
    
    # Get the first match
    product_idx = matching_products.index[0]
    original_price = df.loc[product_idx, 'Price_Numeric']
    
    # Get TF-IDF vector for this product
    product_vector = tfidf_matrix[product_idx]
    
    # Calculate similarity with all products
    similarities = cosine_similarity(product_vector, tfidf_matrix)[0]
    
    # Find products that are cheaper and similar
    cheaper_mask = df['Price_Numeric'] < original_price
    cheaper_products = df[cheaper_mask].copy()
    cheaper_indices = cheaper_products.index.tolist()
    
    # Get similarity scores for cheaper products
    cheaper_similarities = [(idx, similarities[idx]) for idx in cheaper_indices]
    cheaper_similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Get top N results
    top_cheaper = cheaper_similarities[:top_n]
    
    result_indices = [idx for idx, sim in top_cheaper]
    
    results = df.loc[result_indices, ['Product_Name_Extracted', 'Price_Numeric', 'Source', 'Type']].copy()
    results['Similarity_Score'] = [sim for idx, sim in top_cheaper]
    results['Original_Price'] = original_price
    results['Price_Difference'] = results['Original_Price'] - results['Price_Numeric']
    results['Savings_Percentage'] = (results['Price_Difference'] / results['Original_Price'] * 100).round(2)
    
    return results

# Test the model
test_product = 'iQOO Z11x 5G'
print(f"\n{'='*80}")
print(f"Finding cheaper alternatives for: {test_product}")
print(f"{'='*80}\n")
recommendations = find_cheaper_alternatives(test_product, top_n=5)
print(recommendations)



Finding cheaper alternatives for: iQOO Z11x 5G

                               Product_Name_Extracted  Price_Numeric  \
9   iQOO Z10x 5G (Ultramarine, 6GB RAM, 128GB Stor...        16998.0   
97  iQOO Z10x 5G (Ultramarine, 6GB RAM, 128GB Stor...        16998.0   
15  Redmi A5 Jaisalmer Gold (3GB RAM 64GB Storage)...         7999.0   
12  Motorola G57 Power 5G (Corsair, 8GB RAM, 128GB...        15199.0   
16  Motorola G57 Power 5G (Fluidity, 8GB RAM, 128G...        14920.0   

      Source    Type  Similarity_Score  Original_Price  Price_Difference  \
9     Amazon  Mobile          0.936294         18998.0            2000.0   
97  Flipkart  Mobile          0.848005         18998.0            2000.0   
15    Amazon  Mobile          0.727797         18998.0           10999.0   
12    Amazon  Mobile          0.718082         18998.0            3799.0   
16    Amazon  Mobile          0.718082         18998.0            4078.0   

    Savings_Percentage  
9                10.53  
97         

## Step 6: Build the Model Function
Create a reusable function that:
1. Takes a product name as input
2. Finds matching products in the dataset
3. Calculates similarity using cosine distance
4. Returns cheaper alternatives ranked by similarity

In [11]:

# Test with different products
test_products = ['HP Victus', 'Motorola G57 Power 5G', 'POCO C71']

for product in test_products:
    print(f"\n{'='*80}")
    print(f"Cheaper alternatives for: {product}")
    print(f"{'='*80}\n")
    result = find_cheaper_alternatives(product, top_n=1)
    if isinstance(result, str):
        print(result)
    else:
        print(result[['Product_Name_Extracted', 'Price_Numeric', 'Savings_Percentage']])
    print()



Cheaper alternatives for: HP Victus

   Product_Name_Extracted  Price_Numeric  Savings_Percentage
42                  HP 15        42990.0               30.65


Cheaper alternatives for: Motorola G57 Power 5G

   Product_Name_Extracted  Price_Numeric  Savings_Percentage
16  Motorola G57 Power 5G        14920.0                1.84


Cheaper alternatives for: POCO C71

      Product_Name_Extracted  Price_Numeric  Savings_Percentage
5  Samsung Galaxy M07 Mobile         7999.0               19.99



## Step 7: Test with Sample Products
Test the model on real products to see results

In [24]:

# Model Summary
print("\n" + "="*80)
print("MODEL SUMMARY: Find Cheaper Product Alternatives")
print("="*80)
print("""
HOW IT WORKS:
1. Takes a product name as input (e.g., 'iQOO Z11x 5G')
2. Searches for the product in the dataset
3. Uses TF-IDF + Cosine Similarity to find similar products
4. Filters for products that are CHEAPER
5. Returns top N alternatives ranked by similarity

OUTPUT COLUMNS:
- Product_Name_Extracted: Name of the cheaper alternative
- Price_Numeric: Price of the alternative
- Similarity_Score: How similar it is (0-1, higher = more similar)
- Price_Difference: Amount saved in rupees
- Savings_Percentage: Percentage discount compared to original

USAGE:
  find_cheaper_alternatives('iQOO Z11x 5G', top_n=5)
  find_cheaper_alternatives('Samsung Galaxy S26 5G', top_n=3)
  find_cheaper_alternatives('Motorola G57 Power 5G', top_n=7)
""")
print("="*80)



MODEL SUMMARY: Find Cheaper Product Alternatives

HOW IT WORKS:
1. Takes a product name as input (e.g., 'iQOO Z11x 5G')
2. Searches for the product in the dataset
3. Uses TF-IDF + Cosine Similarity to find similar products
4. Filters for products that are CHEAPER
5. Returns top N alternatives ranked by similarity

OUTPUT COLUMNS:
- Product_Name_Extracted: Name of the cheaper alternative
- Price_Numeric: Price of the alternative
- Similarity_Score: How similar it is (0-1, higher = more similar)
- Price_Difference: Amount saved in rupees
- Savings_Percentage: Percentage discount compared to original

USAGE:
  find_cheaper_alternatives('iQOO Z11x 5G', top_n=5)
  find_cheaper_alternatives('Samsung Galaxy S26 5G', top_n=3)
  find_cheaper_alternatives('Motorola G57 Power 5G', top_n=7)



## Step 8: Model Summary & Usage
How to use this model in your own code